In [1]:
from IPython.display import HTML
HTML('''
    <style> body {font-family: "Roboto Condensed Light", "Roboto Condensed";} h2 {padding: 10px 12px; background-color: #E64626; position: static; color: #ffffff; font-size: 40px;} .text_cell_render p { font-size: 15px; } .text_cell_render h1 { font-size: 30px; } h1 {padding: 10px 12px; background-color: #E64626; color: #ffffff; font-size: 40px;} .text_cell_render h3 { padding: 10px 12px; background-color: #0148A4; position: static; color: #ffffff; font-size: 20px;} h4:before{ 
    content: "@"; font-family:"Wingdings"; font-style:regular; margin-right: 4px;} .text_cell_render h4 {padding: 8px; font-family: "Roboto Condensed Light"; position: static; font-style: italic; background-color: #FFB800; color: #ffffff; font-size: 18px; text-align: center; border-radius: 5px;}input[type=submit] {background-color: #E64626; border: solid; border-color: #734036; color: white; padding: 8px 16px; text-decoration: none; margin: 4px 2px; cursor: pointer; border-radius: 20px;}</style>
''')

# Data Cleaning

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt

In [56]:
business_data = pd.read_csv('Businesses.csv')
income_data = pd.read_csv('Income.csv')
polling_data = pd.read_csv('PollingPlaces2019.csv')
population_data = pd.read_csv('Population.csv')
stops_data = pd.read_csv('stops.txt')

primary_school_data = gpd.read_file('Catchments\catchments\catchments_primary.shp')
secondary_school_data = gpd.read_file('Catchments\catchments\catchments_secondary.shp')
future_school_data = gpd.read_file('Catchments\catchments\catchments_future.shp')

## Business Data

In order to simplify the dataset, all the business turnover columns were aggregated into a single column 'total_turnove', which multiplies the number of businesses in each range by that range's median value (taking 12 million for the 10+ column). These values were then summed up to give the areas total_turnover value.

In [6]:
business_data['total_turnover'] = business_data['0_to_50k_businesses'] * 25 + business_data['50k_to_200k_businesses'] * 125 + business_data['200k_to_2m_businesses'] * 1100 + business_data['2m_to_5m_businesses'] * 3500 + business_data['5m_to_10m_businesses'] * 7500 + business_data['10m_or_more_businesses'] * 12000
business_data

,industry_code,industry_name,sa2_code,sa2_name,0_to_50k_businesses,50k_to_200k_businesses,200k_to_2m_businesses,2m_to_5m_businesses,5m_to_10m_businesses,10m_or_more_businesses,total_businesses,total_turnover
0,A,"Agriculture, Forestry and Fishing",101021007,Braidwood,136,92,63,4,0,0,296,98200
1,A,"Agriculture, Forestry and Fishing",101021008,Karabar,6,3,0,0,0,0,9,525
2,A,"Agriculture, Forestry and Fishing",101021009,Queanbeyan,6,4,3,0,0,3,15,39950
3,A,"Agriculture, Forestry and Fishing",101021010,Queanbeyan - East,0,3,0,0,0,0,3,375
4,A,"Agriculture, Forestry and Fishing",101021012,Queanbeyan West - Jerrabomberra,7,4,5,0,0,0,16,6175
...,...,...,...,...,...,...,...,...,...,...,...,...
12212,S,Other Services,128021538,Sutherland - Kirrawee,21,66,58,3,3,0,152,105575
12213,S,Other Services,128021607,Engadine,13,41,31,3,0,0,87,50050
12214,S,Other Services,128021608,Loftus - Yarrawarrah,0,10,10,0,0,0,22,12250
12215,S,Other Services,128021609,Woronora Heights,0,3,5,0,0,0,9,5875


With the new column created, we can drop all of the individual turnover columns, as well as the industry code column (as the information in this column is already captured more effectively in the industry name column).

In [9]:
business_data = business_data[['industry_name','sa2_code','sa2_name','total_businesses','total_turnover']]
business_data.to_csv('Clean_Business.csv')

## Income Data

Looking at the columns here, none seem useless to the analysis and so they are all kept

In [21]:
income_data.head()

,sa2_code21,sa2_name,earners,median_age,median_income,mean_income
0,101021007,Braidwood,2467,51,46640,68904
1,101021008,Karabar,5103,42,65564,69672
2,101021009,Queanbeyan,7028,39,63528,69174
3,101021010,Queanbeyan - East,3398,39,66148,74162
4,101021012,Queanbeyan West - Jerrabomberra,8422,44,78630,91981


## Polling Data

In [55]:
polling_data.head()
sum(polling_data['latitude'].isnull()), len(polling_data)

(140, 2930)

## Population Data

In [14]:
population_data.head()

,sa2_code,sa2_name,0-4_people,5-9_people,10-14_people,15-19_people,20-24_people,25-29_people,30-34_people,35-39_people,...,45-49_people,50-54_people,55-59_people,60-64_people,65-69_people,70-74_people,75-79_people,80-84_people,85-and-over_people,total_people
0,102011028,Avoca Beach - Copacabana,424,522,623,552,386,222,306,416,...,572,602,570,520,464,369,226,142,70,7530
1,102011029,Box Head - MacMasters Beach,511,666,702,592,461,347,420,535,...,749,749,794,895,863,925,603,331,264,11052
2,102011030,Calga - Kulnura,200,225,258,278,274,227,214,286,...,325,436,422,397,327,264,190,100,75,4748
3,102011031,Erina - Green Point,683,804,880,838,661,502,587,757,...,859,882,901,930,917,1065,976,773,1028,14803
4,102011032,Gosford - Springfield,1164,1044,1084,1072,1499,1864,1750,1520,...,1330,1241,1377,1285,1166,949,664,476,537,21346


## Stops Data

In [57]:
stops_data.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
0,200039,200039.0,"Central Station, Eddy Av, Stand A",-33.882206,151.206665,NaN,200060,0,NaN
1,200054,200054.0,"Central Station, Eddy Av, Stand D",-33.882042,151.206991,NaN,200060,0,NaN
2,200060,NaN,Central Station,-33.884084,151.206292,1.0,NaN,0,NaN
3,201510,NaN,Redfern Station,-33.891690,151.198866,1.0,NaN,0,NaN
4,201646,201646.0,"Redfern Station, Gibbons St, Stand B",-33.893329,151.198882,NaN,201510,0,NaN


## Primary Schools

In [34]:
primary_school_data.head()

,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,PRIORITY,geometry
0,2838,PRIMARY,Parklea PS,20181210,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((150.93564 -33.71612, 150.93715 -33.7..."
1,2404,PRIMARY,Lindfield EPS,20211219,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((151.18336 -33.74748, 151.18443 -33.7..."
2,4393,PRIMARY,Carlingford WPS,20220223,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((151.04518 -33.77303, 151.04526 -33.7..."
3,4615,PRIMARY,Caddies Ck PS,20181210,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((150.92567 -33.72960, 150.92602 -33.7..."
4,3918,PRIMARY,Killara PS,20211219,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((151.15379 -33.75586, 151.15404 -33.7..."


From the Primary School data, we have discarded all columns relating to which years the schools cater to. This was done primarily for two reasons. The first of which being the fact that these columns do not accurately portray how 'bustling' a school is. Schools which take fewer years do not necessarily have a smaller student body.
Secondly, these columns served as the most problematic in terms of cleaning, as a numerous entries were mislabelled; to ensure each entry is truly correct would require manual research into each school.

Other removed columns are as follows:
 - 'DATE_ADDED' column was removed as it is unimportant to the analysis of how bustling the SA2 areas are NOW. 
 - 'Priority' column was removed as it is almost entirely Null values, and no description was given for precisely what the column means.
 - 'CATCH_TYPE' column was removed as the specific type of school has little impact on how 'bustling' the school, and surrounding catchment area is.

In [40]:
primary_school_data = primary_school_data[['USE_ID','USE_DESC','geometry']]

,USE_ID,USE_DESC,geometry
0,2838,Parklea PS,"POLYGON ((150.93564 -33.71612, 150.93715 -33.7..."
1,2404,Lindfield EPS,"POLYGON ((151.18336 -33.74748, 151.18443 -33.7..."
2,4393,Carlingford WPS,"POLYGON ((151.04518 -33.77303, 151.04526 -33.7..."
3,4615,Caddies Ck PS,"POLYGON ((150.92567 -33.72960, 150.92602 -33.7..."
4,3918,Killara PS,"POLYGON ((151.15379 -33.75586, 151.15404 -33.7..."
...,...,...,...
1657,4383,E A Southee PS,"POLYGON ((147.94621 -34.55863, 147.95292 -34.5..."
1658,3275,Tumbarumba PS,"POLYGON ((148.12885 -35.60082, 148.23155 -35.6..."
1659,2239,Jindera PS,"POLYGON ((146.86148 -35.87511, 146.87402 -35.8..."
1660,3594,Louth PS,"POLYGON ((145.18403 -29.65805, 145.18434 -29.6..."


The columns we kept are as follows:
 - 'geometry' as this contains the location information, most crucial to the analysis.
 - 'USE_ID' and 'USE_DESC' were kept as two methods of identification for the schools, as whilst they may not play a role in analysis, without them the data becomes far harder to interpret.

## Secondary Schools

In [41]:
secondary_school_data.head()

,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,PRIORITY,geometry
0,8503,HIGH_COED,Billabong HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((146.67182 -35.31444, 146.68930 -35.3..."
1,8266,HIGH_COED,James Fallon HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((147.08734 -35.86271, 147.10413 -35.8..."
2,8505,HIGH_COED,Murray HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((146.81448 -35.78341, 146.81250 -35.7..."
3,8458,HIGH_COED,Kingswood HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"MULTIPOLYGON (((150.68600 -33.74031, 150.68631..."
4,8559,HIGH_COED,Jamison HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((150.69513 -33.75627, 150.68936 -33.7..."


The same columns were removed from the Secondary School data, for the same reason as done to the primary school data.

In [44]:
secondary_school_data = secondary_school_data[['USE_ID','USE_DESC','geometry']]
secondary_school_data

,USE_ID,USE_DESC,geometry
0,8503,Billabong HS,"POLYGON ((146.67182 -35.31444, 146.68930 -35.3..."
1,8266,James Fallon HS,"POLYGON ((147.08734 -35.86271, 147.10413 -35.8..."
2,8505,Murray HS,"POLYGON ((146.81448 -35.78341, 146.81250 -35.7..."
3,8458,Kingswood HS,"MULTIPOLYGON (((150.68600 -33.74031, 150.68631..."
4,8559,Jamison HS,"POLYGON ((150.69513 -33.75627, 150.68936 -33.7..."
...,...,...,...
431,8213,Birrong BHS,"POLYGON ((151.05364 -33.85076, 151.06142 -33.8..."
432,8108,Cessnock HS,"POLYGON ((151.42852 -32.74415, 151.43080 -32.7..."
433,3235,Tooleybuc CS,"POLYGON ((143.37723 -34.80173, 143.39037 -34.8..."
434,1115,Balranald CS,"POLYGON ((143.65541 -33.55702, 143.65541 -33.5..."


## Future Schools

The future school dataset contains only 30 entries, of which many state that they are planned to be operating in 2024 (which would no longer make them future). 
The truth as to whether these schools are up-and-running is not documented on, and so we have decided to not use any of the data from this set.

In [46]:
future_school_data.head()

,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,geometry
0,8416,HIGH_COED,Ku-ring-gai HS,20230114,0,0,0,0,0,0,0,2024,2024,2024,2024,2024,2024,"POLYGON ((151.19849 -33.53990, 151.19945 -33.5..."
1,8161,HIGH_BOYS,Randwick BHS,20200220,0,0,0,0,0,0,0,2024,2024,2024,2024,2024,2024,"POLYGON ((151.27152 -33.91402, 151.27152 -33.9..."
2,8539,HIGH_COED,SSC Blackwattle Bay,20220609,0,0,0,0,0,0,0,0,0,0,0,2024,2024,"POLYGON ((151.15292 -33.83939, 151.16144 -33.8..."
3,8400,HIGH_COED,St Ives HS,20230114,0,0,0,0,0,0,0,2024,2024,2024,2024,2024,2024,"POLYGON ((151.17794 -33.69820, 151.17859 -33.6..."
4,8555,HIGH_COED,Rose Bay SC,20200220,0,0,0,0,0,0,0,2024,2024,2024,2024,2024,2024,"POLYGON ((151.28072 -33.83287, 151.28095 -33.8..."
